In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
            .master("local[*]") \
            .appName("test") \
            .getOrCreate()

25/03/07 15:33:06 WARN Utils: Your hostname, Koustubhs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.183 instead (on interface en0)
25/03/07 15:33:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/07 15:33:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2017-02.parquet

In [3]:
df = spark.read.option("header", "true").parquet("fhv_tripdata_2017-02.parquet")

In [4]:
df.schema

StructType([StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropOff_datetime', TimestampNTZType(), True), StructField('PUlocationID', DoubleType(), True), StructField('DOlocationID', DoubleType(), True), StructField('SR_Flag', IntegerType(), True), StructField('Affiliated_base_number', StringType(), True)])

In [5]:
df.head(5)

[Row(dispatching_base_num='B00008', pickup_datetime=datetime.datetime(2017, 2, 1, 0, 30), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), PUlocationID=None, DOlocationID=None, SR_Flag=None, Affiliated_base_number='B00008'),
 Row(dispatching_base_num='B00008', pickup_datetime=datetime.datetime(2017, 2, 1, 0, 40), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), PUlocationID=None, DOlocationID=None, SR_Flag=None, Affiliated_base_number='B00008'),
 Row(dispatching_base_num='B00009', pickup_datetime=datetime.datetime(2017, 2, 1, 0, 30), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), PUlocationID=None, DOlocationID=None, SR_Flag=None, Affiliated_base_number='B00009'),
 Row(dispatching_base_num='B00013', pickup_datetime=datetime.datetime(2017, 2, 1, 0, 11), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), PUlocationID=None, DOlocationID=None, SR_Flag=None, Affiliated_base_number='B00013'),
 Row(dispatching_base_num='B00013', pickup_datetime=datetime.datetime(2017, 2, 1

Custom schema can be added by reading the parquet file into pandas, getting the dtypes and using the schema in spark.read

Spark df is repartitioned to 4 partitions for efficiency 

In [13]:
df = df.repartition(4)

In [14]:
df.write.parquet('fhv_tripdata/2017/02')

Java HotSpot(TM) 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
Java HotSpot(TM) 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=


CodeCache: size=131072Kb used=17790Kb max_used=17790Kb free=113282Kb
 bounds [0x00000001089e8000, 0x0000000109b68000, 0x00000001109e8000]
 total_blobs=7433 nmethods=6443 adapters=903
 compilation: disabled (not enough contiguous free space left)


Select operations

In [16]:
df.printSchema()

root
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropOff_datetime: timestamp_ntz (nullable = true)
 |-- PUlocationID: double (nullable = true)
 |-- DOlocationID: double (nullable = true)
 |-- SR_Flag: integer (nullable = true)
 |-- Affiliated_base_number: string (nullable = true)



.filter and .select are spark transformations and .show, .take and ,head are spark actions

In [19]:
df.select('pickup_datetime', 'dropOff_datetime', 'SR_Flag').filter(df.dispatching_base_num == 'B00008').head(5)

[Row(pickup_datetime=datetime.datetime(2017, 2, 27, 16, 40), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), SR_Flag=None),
 Row(pickup_datetime=datetime.datetime(2017, 2, 7, 12, 0), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), SR_Flag=None),
 Row(pickup_datetime=datetime.datetime(2017, 2, 8, 8, 30), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), SR_Flag=None),
 Row(pickup_datetime=datetime.datetime(2017, 2, 25, 13, 0), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), SR_Flag=None),
 Row(pickup_datetime=datetime.datetime(2017, 2, 11, 8, 20), dropOff_datetime=datetime.datetime(1989, 1, 1, 0, 0), SR_Flag=None)]

Built in spark functions

In [20]:
from pyspark.sql import functions as F

In [21]:
df.withColumn('pickupdate', F.to_date(df.pickup_datetime)).withColumn('dropoffdate', F.to_date(df.dropOff_datetime)).show()

[Stage 12:==================================================>       (7 + 1) / 8]

+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+----------+-----------+
|dispatching_base_num|    pickup_datetime|   dropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Affiliated_base_number|pickupdate|dropoffdate|
+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+----------+-----------+
|              B00937|2017-02-23 23:27:00|1989-01-01 00:00:00|        NULL|        NULL|   NULL|                B00937|2017-02-23| 1989-01-01|
|              B02914|2017-02-15 11:04:40|1989-01-01 00:00:00|        NULL|        NULL|   NULL|                B02682|2017-02-15| 1989-01-01|
|              B02682|2017-02-04 07:21:20|1989-01-01 00:00:00|       161.0|        NULL|   NULL|                B02682|2017-02-04| 1989-01-01|
|              B02878|2017-02-11 07:07:43|1989-01-01 00:00:00|       181.0|        NULL|   NULL|                B02878|2017-02-11| 1989-01-01|

User defined functions

In [22]:
def udf_spark(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f"s/{num:03x}"
    else:
        return f"e/{num:03x}"

In [23]:
from pyspark.sql import types
custom_udf = F.udf(udf_spark, returnType=types.StringType())
df.withColumn('hex', custom_udf(df.dispatching_base_num)).show()

[Stage 15:==================================================>       (7 + 1) / 8]

+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+-----+
|dispatching_base_num|    pickup_datetime|   dropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Affiliated_base_number|  hex|
+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+-----+
|              B00937|2017-02-23 23:27:00|1989-01-01 00:00:00|        NULL|        NULL|   NULL|                B00937|e/3a9|
|              B02914|2017-02-15 11:04:40|1989-01-01 00:00:00|        NULL|        NULL|   NULL|                B02682|e/b62|
|              B02682|2017-02-04 07:21:20|1989-01-01 00:00:00|       161.0|        NULL|   NULL|                B02682|e/a7a|
|              B02878|2017-02-11 07:07:43|1989-01-01 00:00:00|       181.0|        NULL|   NULL|                B02878|e/b3e|
|              B02764|2017-02-28 18:13:51|1989-01-01 00:00:00|        68.0|        NULL|   NULL|                B00111

Temporary tables can be registered using group by from spark df and written as paruqet files after querying using spark.sql query